In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.inspection import permutation_importance

# Show all columns when printing
pd.set_option('display.max_columns', None)


In [ ]:
target = pd.read_csv('src/data/training/processed/final_training_data.csv')
target['mix__sd_od_history_to_kyc_ratio'] = target['od__history_length_days'] / target['sd__age_since_first_kyc']
target = target.drop(columns=['user_id', 'reference_date', 'default_date'])
target = target.loc[:, ~target.columns.str.startswith('dn__')]

In [ ]:
target

In [ ]:
defref_diff = (
    (pd.to_datetime(target['default_date']).dt.year -
     pd.to_datetime(target['reference_date']).dt.year) * 12
    +
    (pd.to_datetime(target['default_date']).dt.month -
     pd.to_datetime(target['reference_date']).dt.month)
)

target['defref_diff'] = defref_diff

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Example target dataframe: target['defref_diff'], target['ccf']

# Compute counts per month difference
counts = target['defref_diff'].value_counts().sort_index()

# Compute average CCF per month difference
avg_ccf = target.groupby('defref_diff')['t__ccf'].mean()

# Start plotting
fig, ax1 = plt.subplots(figsize=(10,6))

# Bar plot for counts
ax1.bar(counts.index, counts.values, label='Count', color = 'skyblue')
ax1.set_xlabel("Month difference (default_date - reference_date)")
ax1.set_ylabel("Count")
ax1.set_title("Distribution of Default vs Reference Date Difference")

# Secondary y-axis for avg CCF
ax2 = ax1.twinx()
ax2.plot(avg_ccf.index, avg_ccf.values, color='orange', marker='o', label='Average CCF')
ax2.set_ylabel("Average CCF")
ax2.tick_params(axis='y')

# Optional: add legends
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc='upper left')

plt.show()

In [ ]:
sum(counts[:12]) / sum(counts)

In [ ]:
plt.boxplot(target['ob__balance_ref'])

In [ ]:
# Basic stats
print(target['t__ccf'].describe())

# Check for missing values
print(target['t__ccf'].isna().sum())

# Check skewness and kurtosis
print(f"Skewness: {target['t__ccf'].skew()}")
print(f"Kurtosis: {target['t__ccf'].kurt()}")


In [ ]:
# Histogram + KDE
plt.figure(figsize=(10,5))
sns.histplot(target['t__ccf'], bins=50, kde=True, color='skyblue')
plt.title('Distribution of CCF')
plt.show()

# Boxplot to detect outliers
plt.figure(figsize=(8,4))
sns.boxplot(x=target['t__ccf'], color='skyblue')
plt.title('Boxplot of CCF')
plt.show()

# Optionally, check for log-normality
plt.figure(figsize=(10,5))
sns.histplot(np.log1p(target['t__ccf']), bins=50, kde=True, color='skyblue')
plt.title('Log-Transformed Distribution of CCF')
plt.show()

In [ ]:
target['t__ccf'] = target['t__ccf'].clip(upper=2)

In [ ]:
# Basic stats
print(target['t__ccf'].describe())

# Check for missing values
print(target['t__ccf'].isna().sum())

# Check skewness and kurtosis
print(f"Skewness: {target['t__ccf'].skew()}")
print(f"Kurtosis: {target['t__ccf'].kurt()}")

In [ ]:
# Histogram + KDE
plt.figure(figsize=(10,5))
sns.histplot(target['t__ccf'], bins=50, kde=True)
plt.title('Distribution of t__ccf')
plt.show()

# Boxplot to detect outliers
plt.figure(figsize=(8,4))
sns.boxplot(x=target['t__ccf'])
plt.title('Boxplot of t__ccf')
plt.show()

# Optionally, check for log-normality
plt.figure(figsize=(10,5))
sns.histplot(np.log1p(target['t__ccf']), bins=50, kde=True)
plt.title('Log-Transformed Distribution of t__ccf')
plt.show()

In [ ]:
counts = target['t__is_drawn'].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(counts.index.astype(str), counts.values, color='skyblue')
plt.title("Counts of Drawn vs Not Drawn")
plt.xlabel("t__is_drawn")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
features = target.drop(columns=['t__ccf', 't__is_drawn'])
target_ccf = target['t__ccf']
target_is_drawn = target['t__is_drawn']

In [ ]:
def plot_top_features(series, title):
    top = series.head(30).sort_values()
    plt.figure(figsize=(10,6))
    top.plot(kind='barh', color='skyblue')
    plt.title(title)
    plt.xlabel('Importance')
    plt.show()

In [ ]:
# --- 1. Correlation ---
corr_ccf = features.corrwith(target_ccf).abs().sort_values(ascending=False)
corr_is_drawn = features.corrwith(target_is_drawn).abs().sort_values(ascending=False)

In [ ]:
plot_top_features(corr_ccf, 'Top 30 Features by Correlation with t__ccf')

In [ ]:
plot_top_features(corr_is_drawn, 'Top 30 Features by Correlation with t__is_drawn')

In [ ]:
# --- 3. Random Forest Importance ---
rf_ccf = RandomForestRegressor(n_estimators=500, random_state=42)
rf_ccf.fit(features, target_ccf)
rf_ccf_imp = pd.Series(rf_ccf.feature_importances_, index=features.columns).sort_values(ascending=False)

rf_is_drawn = RandomForestClassifier(n_estimators=500, random_state=42)
rf_is_drawn.fit(features, target_is_drawn)
rf_is_drawn_imp = pd.Series(rf_is_drawn.feature_importances_, index=features.columns).sort_values(ascending=False)

In [ ]:
plot_top_features(rf_ccf_imp, 'Top 30 Features by Random Forest Importance (t__ccf)')

In [ ]:
plot_top_features(rf_is_drawn_imp, 'Top 30 Features by Random Forest Importance (t__is_drawn)')

In [ ]:
# --- 4. Permutation Importance ---
perm_ccf = permutation_importance(rf_ccf, features, target_ccf, n_repeats=10, random_state=42)
perm_ccf_imp = pd.Series(perm_ccf.importances_mean, index=features.columns).sort_values(ascending=False)

perm_is_drawn = permutation_importance(rf_is_drawn, features, target_is_drawn, n_repeats=10, random_state=42)
perm_is_drawn_imp = pd.Series(perm_is_drawn.importances_mean, index=features.columns).sort_values(ascending=False)

In [ ]:
plot_top_features(perm_ccf_imp, 'Top 30 Features by Permutation Importance (t__ccf)')

In [ ]:
plot_top_features(perm_is_drawn_imp, 'Top 30 Features by Permutation Importance (t__is_drawn)')

In [ ]:
xgb_ccf = XGBRegressor(random_state=42, n_estimators=1500)
xgb_ccf.fit(features, target_ccf)
xgb_ccf_imp = pd.Series(xgb_ccf.feature_importances_, index=features.columns).sort_values(ascending=False)

In [ ]:
plot_top_features(xgb_ccf_imp, 'Top 30 Features by XGBoost Importance (t__ccf)')

In [ ]:
xgb_is_drawn = XGBClassifier(random_state=42, n_estimators=1500)
xgb_is_drawn.fit(features, target_is_drawn)
xgb_is_drawn_imp = pd.Series(xgb_is_drawn.feature_importances_, index=features.columns).sort_values(ascending=False)

In [ ]:
plot_top_features(xgb_is_drawn_imp, 'Top 30 Features by XGBoost Importance (t__is_drawn)')

In [ ]:
from collections import Counter

def top_features_in_at_least_n(top_lists, n=4):
    all_features = [feat for lst in top_lists for feat in lst]
    counts = Counter(all_features)
    return {feat for feat, c in counts.items() if c >= n}

top_corr_ccf = list(corr_ccf.head(30).index)
top_rf_ccf = list(rf_ccf_imp.head(30).index)
top_perm_ccf = list(perm_ccf_imp.head(30).index)
top_xgb_ccf = list(perm_ccf_imp.head(30).index)

common_ccf_2plus = top_features_in_at_least_n([top_corr_ccf, top_rf_ccf, top_perm_ccf, top_xgb_ccf], n=2)
print("t__ccf features in top 30 of at least 2 metrics:", common_ccf_2plus)

top_corr_is_drawn = list(corr_is_drawn.head(30).index)
top_rf_is_drawn = list(rf_is_drawn_imp.head(30).index)
top_perm_is_drawn = list(perm_is_drawn_imp.head(30).index)
top_xgb_is_drawn = list(xgb_is_drawn_imp.head(30).index)

common_is_drawn_2plus = top_features_in_at_least_n([top_corr_is_drawn, top_rf_is_drawn, top_perm_is_drawn, top_xgb_is_drawn], n=2)
print("t__is_drawn features in top 30 of at least 2 metrics:", common_is_drawn_2plus)


In [ ]:
['ob__avg_util_0_1m', 'ob__limit_ref']

In [ ]:
# --- Top 20 sets for t__ccf ---
top_corr_ccf = set(corr_ccf.head(20).index)
top_rf_ccf = set(rf_ccf_imp.head(20).index)
top_perm_ccf = set(perm_ccf_imp.head(20).index)

# Intersection across all 4 methods
common_ccf = top_corr_ccf & top_mi_ccf & top_rf_ccf & top_perm_ccf
print("Features consistently important for t__ccf:", common_ccf)

In [ ]:
# --- Top 20 sets for t__is_drawn ---
top_corr_is_drawn = set(corr_is_drawn.head(20).index)
top_mi_is_drawn = set(mi_is_drawn.head(20).index)
top_rf_is_drawn = set(rf_is_drawn_imp.head(20).index)
top_perm_is_drawn = set(perm_is_drawn_imp.head(20).index)

# Intersection across all 4 methods
common_is_drawn = top_corr_is_drawn & top_mi_is_drawn & top_rf_is_drawn & top_perm_is_drawn
print("Features consistently important for t__is_drawn:", common_is_drawn)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

target['default_date'] = pd.to_datetime(target['default_date'])
target['eom_default'] = target['default_date'] + pd.offsets.MonthEnd(0)

agg = (
    target
    .groupby('eom_default')
    .agg(
        avg_ccf=('t__ccf', 'mean'),
        n_obs=('t__ccf', 'size')
    )
    .reset_index()
    .sort_values('eom_default')
)

# convert to string for plotting
agg['eom_str'] = agg['eom_default'].dt.to_period('M').astype(str)


In [ ]:
agg

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
import numpy as np

# Example model periods
version_changes = {
    '2019-09': 'lisbon_v1',
    '2022-04': 'lisbon_v2',
    '2023-04': 'lisbon_v3',
    '2023-09': 'lisbon_v4',
    '2024-01': 'porto_v1',
    '2024-07': 'porto_v2'
}
# Assume agg already prepared
agg['eom_str'] = agg['eom_default'].dt.to_period('M').astype(str)

fig, ax1 = plt.subplots(figsize=(12, 5))

# --- bars ---
ax1.bar(agg['eom_str'], agg['n_obs'], color='skyblue')
ax1.set_ylabel('Number of defaults')
ax1.set_xlabel('Default month (EOM)')

# show only every 3rd tick
ticks_to_show = agg['eom_str'].iloc[::3]
ax1.set_xticks(ticks_to_show)
ax1.set_xticklabels(ticks_to_show, rotation=45)

# --- line: avg CCF ---
ax2 = ax1.twinx()
ax2.plot(agg['eom_str'], agg['avg_ccf'], marker='o', color='orange')
ax2.set_ylabel('Average CCF')

# --- add vertical lines and text for version changes ---
for eom, label in version_changes.items():
    if eom in agg['eom_str'].values:
        ax1.axvline(x=eom, color='red', linestyle='--', linewidth=1)
        ax1.text(
            eom, 
            ax1.get_ylim()[1]*0.95,  # slightly below top of y-axis
            label, 
            color='red', 
            rotation=90, 
            va='top', 
            ha='right',
            fontsize=9
        )

plt.title('Average CCF and Number of Defaults by Default Month')
plt.tight_layout()
plt.show()